In [ ]:
import sys
import os
import logging

sys.path.insert(0, '.')

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")

from shared.data_loader import load_stage5_horizon5
from shared.trainer import train_and_evaluate, save_results_json
from shared.plots import (
    plot_roc_curves,
    plot_confusion_matrices,
    plot_calibration_diagrams,
    plot_temporal_stability,
    plot_prediction_distributions,
    plot_learning_curves,
    plot_feature_importance,
    plot_shap_summary,
)

print("Modulos carregados com sucesso")

# Stage 5b: Horizonte 5 dias — Avaliacao Completa

Retreino dos 7 modelos com features engenheiradas de sentimento + preco. Dataset: ~1227 dias, 32 features, horizonte 5 dias.

In [ ]:
data = load_stage5_horizon5(window=30)
print(f"Descricao: {data['description']}")
print(f"Sequencias: {data['X_seq'].shape}")
print(f"Tabular: {data['X_flat'].shape}")
print(f"Features: {data['feature_names']}")
print(f"Balance: {data['y_seq'].mean():.1%} sobe / {1-data['y_seq'].mean():.1%} desce")

## Treino dos 7 modelos (configuracao padrao)

In [ ]:
MODEL_CONFIGS = {
    "BiLSTM Original": {"model_name": "bilstm_original"},
    "BiLSTM Reduzido": {"model_name": "bilstm_reduced"},
    "Transformer": {"model_name": "transformer"},
    "TCN": {"model_name": "tcn"},
    "XGBoost": {"model_name": "xgboost"},
    "Logistic Regression": {"model_name": "logistic_regression"},
    "Random Forest": {"model_name": "random_forest"},
}

results = {}
for display_name, config in MODEL_CONFIGS.items():
    print(f"
{'='*60}")
    print(f"Treinando: {display_name}")
    print(f"{'='*60}")
    results[display_name] = train_and_evaluate(
        model_name=config["model_name"],
        data=data,
        compute_shap=True,
        compute_learning_curve=True,
    )

## Tabela comparativa

In [ ]:
import pandas as pd

rows = []
for name, r in results.items():
    rows.append({
        "Modelo": name,
        "ROC-AUC": f"{r['classification']['roc_auc']:.4f}",
        "Accuracy": f"{r['classification']['accuracy']:.1%}",
        "F1 (Sobe)": f"{r['classification']['f1']:.4f}",
        "Precision (Sobe)": f"{r['classification']['precision']:.4f}",
        "Recall (Sobe)": f"{r['classification']['recall']:.4f}",
        "F1 (Desce)": f"{r['classification']['f1_desce']:.4f}",
        "ECE": f"{r['calibration']['ece']:.4f}",
        "Brier": f"{r['calibration']['brier_score']:.4f}",
        "Tempo (s)": f"{r['train_time_seconds']:.1f}",
    })

df_results = pd.DataFrame(rows)
print("
" + "="*80)
print("TABELA COMPARATIVA — Stage 5b: Horizonte 5 dias")
print("="*80)
display(df_results)

## Diagnosticos visuais

In [ ]:
plot_roc_curves(results, title="Curvas ROC — Stage 5b: Horizonte 5 dias",
                save_path="results/stage5_roc.png")

In [ ]:
plot_confusion_matrices(
    {name: r["classification"] for name, r in results.items()},
    save_path="results/stage5_confusion.png",
)

In [ ]:
plot_calibration_diagrams(
    {name: r["calibration"] for name, r in results.items()},
    save_path="results/stage5_calibration.png",
)

In [ ]:
plot_temporal_stability(
    {name: r["temporal_stability"] for name, r in results.items()},
    save_path="results/stage5_temporal.png",
)

In [ ]:
plot_prediction_distributions(
    {name: r["prediction_distribution"] for name, r in results.items()},
    save_path="results/stage5_distributions.png",
)

## Interpretabilidade (SHAP + Permutation Importance)

In [ ]:
for name in ["XGBoost", "Logistic Regression", "Random Forest"]:
    r = results[name]
    if r["shap_values"] is not None:
        plot_shap_summary(
            r["shap_values"], data["feature_names"],
            title=f"SHAP — {name}",
            save_path=f"results/stage5_shap_{name.lower().replace(' ','_')}.png",
        )
    if r["permutation_importance"] is not None:
        plot_feature_importance(
            r["permutation_importance"],
            r["classification"]["roc_auc"],
            title=f"Permutation Importance — {name}",
            save_path=f"results/stage5_perm_{name.lower().replace(' ','_')}.png",
        )

## Curvas de aprendizado

In [ ]:
lc_results = {name: r["learning_curve"] for name, r in results.items()
               if r["learning_curve"] is not None}
if lc_results:
    plot_learning_curves(lc_results, save_path="results/stage5_learning_curves.png")

## Variacoes de hiperparametros

In [ ]:
VARIATIONS = {
    "Transformer d=32": {"model_name": "transformer", "model_params": {"d_model": 32, "nhead": 2}},
    "Transformer d=128": {"model_name": "transformer", "model_params": {"d_model": 128, "nhead": 8}},
    "Transformer 4L": {"model_name": "transformer", "model_params": {"n_layers": 4}},
    "XGBoost depth=3": {"model_name": "xgboost", "model_params": {"max_depth": 3}},
    "XGBoost depth=6": {"model_name": "xgboost", "model_params": {"max_depth": 6}},
    "XGBoost 500 trees": {"model_name": "xgboost", "model_params": {"n_estimators": 500}},
    "BiLSTM drop=0.1": {"model_name": "bilstm_original", "model_params": {"dropout": 0.1}},
    "BiLSTM drop=0.5": {"model_name": "bilstm_original", "model_params": {"dropout": 0.5}},
    "BiLSTM h=256": {"model_name": "bilstm_original", "model_params": {"hidden_size": 256}},
    "TCN k=2": {"model_name": "tcn", "model_params": {"kernel_size": 2}},
    "TCN k=5": {"model_name": "tcn", "model_params": {"kernel_size": 5}},
    "TCN [32,32]": {"model_name": "tcn", "model_params": {"num_channels": [32, 32]}},
    "LR C=0.01": {"model_name": "logistic_regression", "model_params": {"C": 0.01}},
    "LR C=100": {"model_name": "logistic_regression", "model_params": {"C": 100}},
    "LR L1": {"model_name": "logistic_regression", "model_params": {"penalty": "l1", "solver": "saga"}},
    "RF depth=5": {"model_name": "random_forest", "model_params": {"max_depth": 5}},
    "RF depth=20": {"model_name": "random_forest", "model_params": {"max_depth": 20}},
    "RF 500 trees": {"model_name": "random_forest", "model_params": {"n_estimators": 500}},
}

var_results = {}
for display_name, config in VARIATIONS.items():
    print(f"
Variacao: {display_name}")
    var_results[display_name] = train_and_evaluate(
        model_name=config["model_name"],
        data=data,
        model_params=config.get("model_params", {}),
        compute_shap=False,
        compute_learning_curve=False,
    )

In [ ]:
var_rows = []
for name, r in var_results.items():
    var_rows.append({
        "Variacao": name,
        "ROC-AUC": f"{r['classification']['roc_auc']:.4f}",
        "Accuracy": f"{r['classification']['accuracy']:.1%}",
        "F1": f"{r['classification']['f1']:.4f}",
        "ECE": f"{r['calibration']['ece']:.4f}",
    })

df_var = pd.DataFrame(var_rows)
print("
" + "="*60)
print("VARIACOES DE HIPERPARAMETROS")
print("="*60)
display(df_var)

## Salvar resultados

In [ ]:
all_results = {**results, **var_results}
save_results_json(all_results, "results/stage5_metrics.json")
print("Resultados salvos em results/stage5_metrics.json")